<a href="https://colab.research.google.com/github/laurakeita/MMM/blob/main/notebooks/06_attribution_validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 06 · Attribution Validation
Compare Meridian's estimated ROI against the known ground-truth ROI used to generate the synthetic dataset.

*Requires `mmm` and `media_summary` from `05_synthetic_validation` + `03_model_diagnostics`.*

## Ground-Truth Setup

Notebook 05 generated a synthetic dataset with **known** ROI parameters:

| Channel | True ROI |
|---------|----------|
| Meta    | 2.07     |
| Google  | 3.00     |

TikTok was added to the real dataset but is absent from the synthetic ground-truth because it was not part of the 2-channel synthetic generation in notebook 05.

The goal here is to measure how accurately Meridian recovers these known values — a standard sanity check for Bayesian MMM models.

**Key question:** Is the posterior estimate close enough to the true value to trust the model on real (unobservable) data?

In [ ]:
# ==========================================
# Attribution Validation
# ==========================================

print("\n" + "="*50)
print("ATTRIBUTION VALIDATION")
print("="*50)

# Ground Truth ROI from synthetic dataset
true_roi = {
    "Meta": 2.07,
    "Google": 3.00
}

# Meridian estimated ROI
summary_df = media_summary.summary_table()

print("\nEstimated ROI:")
print(summary_df)

roi_validation_results = []

for channel in true_roi.keys():

    try:
        # Filter for posterior distribution for the current channel
        channel_posterior_summary = summary_df[
            (summary_df["distribution"] == "posterior") &
            (summary_df["channel"].str.contains(channel, case=False, na=False))
        ]

        if not channel_posterior_summary.empty:
            roi_str = channel_posterior_summary["roi"].iloc[0]
            estimated_roi = float(roi_str.split(' ')[0])

            error_pct = (
                abs(estimated_roi - true_roi[channel])
                / true_roi[channel]
            ) * 100

            roi_validation_results.append({
                "channel": channel,
                "true_roi": true_roi[channel],
                "estimated_roi": estimated_roi,
                "recovery_error_pct": round(error_pct,2)
            })
        else:
            print(f"Could not find posterior ROI for {channel}")

    except Exception as e:
        print(f"An error occurred while processing ROI for {channel}: {e}")

validation_df = pd.DataFrame(
    roi_validation_results
)

print("\nROI Recovery Validation")
print(validation_df)

avg_error = validation_df[
    "recovery_error_pct"
].mean()

print(
    f"\nAverage ROI Recovery Error: {avg_error:.2f}%"
)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# ==========================================
# ROI Recovery Bar Chart
# ==========================================

channels = validation_df["channel"].tolist()
true_vals = validation_df["true_roi"].tolist()
est_vals  = validation_df["estimated_roi"].tolist()
errors    = validation_df["recovery_error_pct"].tolist()

x = np.arange(len(channels))
width = 0.35

fig, ax = plt.subplots(figsize=(7, 5))
bars_true = ax.bar(x - width/2, true_vals, width, label="Ground-Truth ROI",
                   color="#4ECDE6", edgecolor="white")
bars_est  = ax.bar(x + width/2, est_vals,  width, label="Meridian Estimated ROI",
                   color="#1A73E8", edgecolor="white")

# Annotate bars with values
for bar, val in zip(bars_true, true_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            f"{val:.2f}", ha="center", va="bottom", fontsize=10, color="#202124")
for bar, val, err in zip(bars_est, est_vals, errors):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            f"{val:.2f}", ha="center", va="bottom", fontsize=10, color="#202124")
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.28,
            f"err {err:.0f}%", ha="center", va="bottom", fontsize=8, color="#d93025")

ax.set_xticks(x)
ax.set_xticklabels(channels, fontsize=12)
ax.set_ylabel("ROI", fontsize=11)
ax.set_title("True vs Estimated ROI — Ground-Truth Recovery", fontsize=13, pad=14)
ax.set_ylim(0, max(true_vals) * 1.35)
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)

avg_err = validation_df["recovery_error_pct"].mean()
fig.text(0.5, -0.02,
         f"Average recovery error: {avg_err:.1f}%  ·  Meta within 15%, Google under-estimated by ~37% (typical for high-ROI channels with limited spend variation)",
         ha="center", fontsize=9, color="#5F6368")

plt.tight_layout()
plt.show()
print(f"\nAverage ROI Recovery Error: {avg_err:.1f}%")